# 02 - Feature Engineering

This notebook constructs cycle-level engineered features from discharge and charge profiles, then exports the full feature table for modeling.

In [1]:
from pathlib import Path
import sys
import random
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
METRICS_PATH = PROJECT_ROOT / 'results' / 'metrics.csv'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## Load data and generate engineered features

In [2]:
from src.data_loader import DEFAULT_BATTERY_IDS, download_battery_mat_files
from src.preprocessing import run_preprocessing_pipeline
from src.features import engineer_features_from_detailed

_ = download_battery_mat_files(RAW_DIR, DEFAULT_BATTERY_IDS)
_, detailed_df = run_preprocessing_pipeline(RAW_DIR, PROCESSED_DIR, DEFAULT_BATTERY_IDS)
features_df = engineer_features_from_detailed(detailed_df)
features_df.to_csv(PROCESSED_DIR / 'features_all.csv', index=False)

print('features_all.csv saved at:', PROCESSED_DIR / 'features_all.csv')
print('Shape:', features_df.shape)
features_df.head()

features_all.csv saved at: /Users/parsa/Engineering_EV_Battery_Degradation/data/processed/features_all.csv
Shape: (636, 24)


,battery_id,cycle_number,capacity_Ah,avg_voltage,avg_current,avg_temperature,charge_time,discharge_time,energy_charged_Wh,SOH,...,max_temperature,temperature_rise,discharge_duration,energy_discharged,internal_resistance_proxy,capacity_fade_rate,cycle_efficiency,capacity_lag_1,capacity_lag_3,capacity_lag_5
0,B0005,1,1.862203,3.529829,1.818712,32.572328,7597.875,3690.234,3.276268,93.110144,...,38.982181,14.652147,3690.234,6.608778,0.216948,-0.004175,2.017167,1.862203,1.862203,1.862203
1,B0005,2,1.852078,3.537320,1.817644,32.725235,10516.000,3672.344,7.638814,92.603889,...,39.033398,14.335646,3672.344,6.586345,0.990131,-0.004175,0.862221,1.862203,1.862203,1.862203
2,B0005,3,1.841049,3.543737,1.816542,32.642862,10484.547,3651.641,7.608577,92.052437,...,38.818797,14.084531,3651.641,6.555683,26.274884,-0.004175,0.861618,1.852078,1.862203,1.862203
3,B0005,4,1.840912,3.543666,1.825618,32.514876,10397.890,3631.563,7.578063,92.045600,...,38.762305,14.108068,3631.563,6.554829,0.235616,-0.004175,0.864974,1.841049,1.862203,1.862203
4,B0005,5,1.840360,3.542343,1.826148,32.382349,10495.203,3629.172,7.565826,92.017996,...,38.665393,14.140596,3629.172,6.552232,0.094378,-0.004175,0.866030,1.840912,1.852078,1.862203


## Quick feature sanity checks

In [3]:
required_feature_cols = [
    'max_voltage',
    'min_voltage',
    'voltage_range',
    'max_temperature',
    'temperature_rise',
    'discharge_duration',
    'energy_discharged',
    'internal_resistance_proxy',
    'capacity_fade_rate',
    'cycle_efficiency',
    'capacity_lag_1',
    'capacity_lag_3',
    'capacity_lag_5',
]

missing = sorted(set(required_feature_cols) - set(features_df.columns))
print('Missing engineered features:', missing)
features_df[required_feature_cols].describe().T

Missing engineered features: []


,count,mean,std,min,25%,50%,75%,max
max_voltage,636.0,4.191865,0.008575,4.157552,4.185758,4.190569,4.199236,4.233325
min_voltage,636.0,2.387737,0.219906,1.737030,2.186663,2.421034,2.529182,2.699983
voltage_range,636.0,1.804128,0.222127,1.493270,1.659363,1.762337,2.012246,2.459612
max_temperature,636.0,39.571456,1.438533,36.372088,38.370834,39.671952,40.870487,42.332522
temperature_rise,636.0,15.623465,1.232751,12.993038,14.589946,15.647367,16.750586,17.655021
discharge_duration,636.0,3116.977701,242.197224,2742.843000,2891.996250,3084.281000,3311.828000,3690.234000
energy_discharged,636.0,5.593139,0.771215,3.924465,4.980789,5.506337,6.295511,7.259533
internal_resistance_proxy,636.0,1.992048,25.355728,0.000041,0.040648,0.115962,0.300998,542.527786
capacity_fade_rate,636.0,-0.003821,0.005955,-0.024107,-0.006362,-0.004301,-0.001958,0.027329
cycle_efficiency,636.0,1.270704,6.035326,0.792786,0.836061,0.852567,0.865487,136.191355
